# Phân tích các yếu tố ảnh hưởng đến rating người dùng trên MovieLens bằng cách kết hợp dữ liệu thông tin phim (TMDb)

In [21]:
import pandas as pd
import numpy as np
import re
import json


## Chuẩn bị và Làm sạch Dữ liệu (Data Preparation)

In [22]:
# --- 1. ĐỌC BỘ DỮ LIỆU MOVIELENS 1M ---
# Định nghĩa tên cột cho các file .dat
u_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
m_cols = ['MovieID', 'Title', 'Genres']
r_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']

# Đọc file users.dat
users = pd.read_csv('datasets/movie_len_1M/users.dat', sep='::', names=u_cols, encoding='latin-1', engine='python')

# Đọc file movies.dat
movies = pd.read_csv('datasets/movie_len_1M/movies.dat', sep='::', names=m_cols, encoding='latin-1', engine='python')

# Đọc file ratings.dat
ratings = pd.read_csv('datasets/movie_len_1M/ratings.dat', sep='::', names=r_cols, encoding='latin-1', engine='python')

print("--- Đã tải xong MovieLens 1M ---")
print(f"Users shape: {users.shape}")
print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")
print("-" * 30)

--- Đã tải xong MovieLens 1M ---
Users shape: (6040, 5)
Movies shape: (3883, 3)
Ratings shape: (1000209, 4)
------------------------------


In [23]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [24]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [25]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [26]:
# --- 2. ĐỌC BỘ DỮ LIỆU TMDB 5000 ---
# TMDB thường sử dụng chuẩn UTF-8, không phải latin-1. 
# Nếu gặp lỗi dòng này, hãy thử xóa tham số encoding (để pandas tự nhận diện)
try:
    tmdb_credits = pd.read_csv('datasets/tmdb_5000_credits.csv', encoding='utf-8')
except UnicodeDecodeError:
    # Fallback nếu file thực sự dùng định dạng cũ (ít gặp với bộ TMDB này)
    tmdb_credits = pd.read_csv('datasets/tmdb_5000_credits.csv', encoding='latin-1')

print("--- Đã tải xong TMDB 5000 Credits ---")
print(f"TMDB Credits shape: {tmdb_credits.shape}")
print(f"Các cột trong TMDB: {list(tmdb_credits.columns)}")

--- Đã tải xong TMDB 5000 Credits ---
TMDB Credits shape: (4803, 4)
Các cột trong TMDB: ['movie_id', 'title', 'cast', 'crew']


In [27]:
tmdb_credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [28]:
# --- 3. HIỂN THỊ DỮ LIỆU MẪU ĐỂ KIỂM TRA ---

print("\n--- Sample Users ---")
users.head(3)

print("\n--- Sample Movies ---")
movies.head(3)

print("\n--- Sample Ratings ---")
ratings.head(3)

print("\n--- Sample TMDB Credits ---")
tmdb_credits.head(3)


--- Sample Users ---

--- Sample Movies ---

--- Sample Ratings ---

--- Sample TMDB Credits ---


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


### Chuẩn hóa

In [29]:
# --- 3. XỬ LÝ LÀM SẠCH TÊN PHIM (PRE-PROCESSING) ---

def clean_movielens_title(text):
    """
    Hàm này loại bỏ năm phát hành dạng (YYYY) ở cuối chuỗi
    và chuyển về chữ thường để so sánh.
    Ví dụ: "Toy Story (1995)" -> "toy story"
    """
    text = str(text)
    # Regex tìm khoảng trắng + mở ngoặc + 4 số + đóng ngoặc ở cuối dòng
    text = re.sub(r'\s*\(\d{4}\)$', '', text) 
    return text.lower().strip()

def clean_tmdb_title(text):
    """
    Chuyển tên phim TMDB về chữ thường và bỏ khoảng trắng thừa
    Ví dụ: "Toy Story " -> "toy story"
    """
    return str(text).lower().strip()

# Tạo cột phụ 'title_join' để làm khóa nối
movies['title_join'] = movies['Title'].apply(clean_movielens_title)
tmdb_credits['title_join'] = tmdb_credits['title'].apply(clean_tmdb_title)

# --- 4. THỰC HIỆN MERGE (GHÉP BẢNG) ---

# Dùng inner join: Chỉ giữ lại những phim có mặt ở CẢ 2 bộ dữ liệu
# Nếu phim nào bên MovieLens không tìm thấy bên TMDB (hoặc ngược lại) sẽ bị bỏ qua
movies_merged = pd.merge(movies, tmdb_credits, on='title_join', how='inner')

# --- 5. KIỂM TRA KẾT QUẢ ---
print("-" * 30)
print(f"Số lượng phim gốc (MovieLens): {movies.shape[0]}")
print(f"Số lượng phim gốc (TMDB): {tmdb_credits.shape[0]}")
print(f"-> Số lượng phim khớp nhau thành công: {movies_merged.shape[0]}")
print("-" * 30)

# Xem thử 5 dòng đầu tiên của bảng đã ghép
# Chú ý: Bảng này giờ có cả 'MovieID' (của ML) và cast/crew (của TMDB)
print(movies_merged[['MovieID', 'Title', 'title', 'cast', 'crew']].head())

------------------------------
Số lượng phim gốc (MovieLens): 3883
Số lượng phim gốc (TMDB): 4803
-> Số lượng phim khớp nhau thành công: 892
------------------------------
   MovieID                    Title             title  \
0        1         Toy Story (1995)         Toy Story   
1       10         GoldenEye (1995)         GoldenEye   
2       14             Nixon (1995)             Nixon   
3       15  Cutthroat Island (1995)  Cutthroat Island   
4       16            Casino (1995)            Casino   

                                                cast  \
0  [{"cast_id": 14, "character": "Woody (voice)",...   
1  [{"cast_id": 1, "character": "James Bond", "cr...   
2  [{"cast_id": 1, "character": "Richard Nixon", ...   
3  [{"cast_id": 1, "character": "Morgan Adams", "...   
4  [{"cast_id": 4, "character": "Sam 'Ace' Rothst...   

                                                crew  
0  [{"credit_id": "52fe4284c3a36847f8024f55", "de...  
1  [{"credit_id": "52fe426ec3a36847f80

### Tạo đặc trưng

In [30]:
# Hàm lấy tên đạo diễn từ chuỗi JSON
def get_director(crew_data):
    try:
        # Chuyển chuỗi text thành list các dictionary
        crew_list = json.loads(crew_data.replace("'", "\"")) # Xử lý dấu nháy đơn nếu cần
    except:
        return None
        
    for crew_member in crew_list:
        if crew_member['job'] == 'Director':
            return crew_member['name']
    return None

# Hàm lấy top 3 diễn viên
def get_top3_actors(cast_data):
    try:
        cast_list = json.loads(cast_data.replace("'", "\""))
    except:
        return []
        
    # Lấy tên của tối đa 3 người đầu tiên
    names = [member['name'] for member in cast_list[:3]]
    return names

# --- ÁP DỤNG VÀO DATAFRAME ---
# Lưu ý: Cột cast/crew của TMDB đôi khi dùng định dạng chuỗi khó chịu, 
# ta dùng hàm apply đơn giản, nếu lỗi thì dùng thư viện ast.literal_eval chuẩn hơn.
import ast

def safe_extract_director(x):
    try:
        data = ast.literal_eval(x)
        for item in data:
            if item['job'] == 'Director':
                return item['name']
    except:
        return None
    return None

def safe_extract_actors(x):
    try:
        data = ast.literal_eval(x)
        names = [item['name'] for item in data]
        return names[:3] # Lấy 3 người đầu
    except:
        return []

print("Đang trích xuất dữ liệu (việc này có thể mất vài giây)...")
movies_merged['director'] = movies_merged['crew'].apply(safe_extract_director)
movies_merged['actors'] = movies_merged['cast'].apply(safe_extract_actors)

# Tách 3 diễn viên ra 3 cột riêng để dễ group
movies_merged['actor_1'] = movies_merged['actors'].apply(lambda x: x[0] if len(x) > 0 else None)
movies_merged['actor_2'] = movies_merged['actors'].apply(lambda x: x[1] if len(x) > 1 else None)
movies_merged['actor_3'] = movies_merged['actors'].apply(lambda x: x[2] if len(x) > 2 else None)

print(movies_merged[['Title', 'director', 'actor_1']].head())

Đang trích xuất dữ liệu (việc này có thể mất vài giây)...
                     Title         director          actor_1
0         Toy Story (1995)    John Lasseter        Tom Hanks
1         GoldenEye (1995)  Martin Campbell   Pierce Brosnan
2             Nixon (1995)     Oliver Stone  Anthony Hopkins
3  Cutthroat Island (1995)     Renny Harlin      Geena Davis
4            Casino (1995)  Martin Scorsese   Robert De Niro


In [31]:
# Nối bảng phim (đã merge) với bảng ratings
# Lưu ý: Bảng ratings rất lớn (1M dòng), việc merge sẽ tạo ra dataframe lớn
full_data = pd.merge(ratings, movies_merged, on='MovieID', how='inner')

print(f"Tổng số dòng dữ liệu để phân tích: {full_data.shape[0]}")

Tổng số dòng dữ liệu để phân tích: 450575


## Khám phá Dữ liệu (EDA - Exploratory Data Analysis)

## Phân tích Chuyên sâu - Các yếu tố ảnh hưởng Rating

+ Tác động của Đạo diễn (The Director's Cut)
+ Sức hút của Diễn viên (Star Power)

### 1. Yếu tố Tài chính (Ngân sách, Doanh thu)

Yếu tố tài chính ảnh hưởng đến Rating như thế nào? 

Ngân sách (Budget) hoặc Doanh thu (Revenue) có mối liên hệ tích cực hay tiêu cực đối với rating trung bình của phim?